# 03. Model Design — Baseline (M1): Logistic Regression

**Judul proyek:** Deteksi Dini Risiko Kecemasan Sedang–Berat pada Gamer Online

**Fokus lifecycle:** Bab 6.1 — Baseline Model Design, Training, dan Validation

Notebook ini melatih satu baseline yang transparan sebagai pembanding adil bagi model advanced. Semua angka evaluasi pada notebook ini berasal dari **validation set**, bukan hold-out test set.

## 6.1.1 Pilihan model dan justifikasi

Model baseline adalah **Logistic Regression berbobot kelas**. Model ini cocok sebagai titik awal karena: (1) target bersifat biner (`risiko_tinggi`: GAD-7 ≥ 10), (2) ukuran data cukup besar tetapi jumlah fitur setelah one-hot encoding relatif kecil, (3) pengaruh setiap fitur dapat ditafsirkan melalui arah/koefisien, dan (4) probabilitas risiko dapat langsung digunakan untuk ROC-AUC serta PR-AUC.

Karakteristik data yang menjadi pertimbangan: kelas positif hanya sekitar 17,4%, terdapat gabungan fitur numerik dan kategorikal, serta hubungan faktor psikologis dengan kecemasan diduga terutama bersifat aditif pada skala log-odds. Karena itu `class_weight='balanced'` dipakai agar kelas risiko tinggi tidak terabaikan.

### Teori, hypothesis space, dan inductive bias

Logistic Regression memodelkan probabilitas kelas positif sebagai $P(y=1|x)=\sigma(w^Tx+b)$, dengan $\sigma$ adalah fungsi sigmoid. Keputusan default memakai ambang 0,50.

- **Hypothesis space:** seluruh pemisah keputusan linear pada fitur hasil transformasi (fitur numerik terskala dan indikator one-hot). Model dapat memberi bobot berbeda pada setiap kategori, tetapi tidak otomatis belajar interaksi non-linear.
- **Inductive bias:** contoh dengan kombinasi fitur yang sama akan memiliki risiko serupa; tiap fitur memengaruhi *log-odds* secara aditif dan linear; regularisasi L2 bawaan menahan koefisien agar tidak terlalu ekstrem.
- **Batasan:** pola ambang dan interaksi seperti `Hours × SPIN_T` tidak tertangkap kecuali direkayasa secara eksplisit. Ini memang disengaja untuk baseline yang sederhana dan mudah diaudit.

## 6.1.2 Import dan konfigurasi reproduktibilitas

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f'Random seed: {RANDOM_SEED}')

Random seed: 42


## 6.1.3 Memuat data dan menjaga pemisahan data

Urutan split adalah 80:20 untuk hold-out test, kemudian 75:25 dari bagian train+validation. Hasil akhirnya 60% train, 20% validation, 20% hold-out test. Test disimpan dan **tidak digunakan** untuk memilih model atau hyperparameter. Kolom GAD item, total GAD, dan evaluasi kesulitan GAD dikeluarkan karena merupakan label leakage.

In [2]:
data_path = Path('../data/GamingStudy_data.csv')
if not data_path.exists():
    data_path = Path('data/GamingStudy_data.csv')

df = pd.read_csv(data_path, encoding='latin1')
df = df[(df['accept'] == 'Accept') & df['Hours'].notna() & (df['Hours'] <= 84)].copy()
y = (df['GAD_T'] >= 10).astype(int)

leakage_cols = ['GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'GAD_T', 'GADE']
uninformative_cols = ['S. No.', 'Timestamp', 'accept', 'highestleague', 'League',
                      'Birthplace', 'Residence', 'Reference', 'Residence_ISO3', 'Birthplace_ISO3']
X = df.drop(columns=leakage_cols + uninformative_cols).copy()

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=RANDOM_SEED
)

print(f'Ukuran data setelah filter: {len(df):,}')
print(f'Train: {len(X_train):,} | Validation: {len(X_val):,} | Hold-out test: {len(X_test):,}')
print(f'Proporsi positif — train: {y_train.mean():.2%}, validation: {y_val.mean():.2%}, test: {y_test.mean():.2%}')

Ukuran data setelah filter: 12,981
Train: 7,788 | Validation: 2,596 | Hold-out test: 2,597
Proporsi positif — train: 17.39%, validation: 17.37%, test: 17.40%


## 6.1.4 Preprocessing pipeline

Pipeline dipelajari hanya dari `X_train`: median + RobustScaler untuk numerik, serta modus + one-hot encoding untuk kategorikal. Hal ini mencegah statistik validation/test masuk ke proses pelatihan. Kategori game, gaya bermain, dan alasan bermain yang jarang digabung menjadi `Other` untuk mengurangi sparsity.

In [3]:
top_games = ['League of Legends', 'Counter Strike', 'Starcraft 2', 'World of Warcraft', 'Other']
X_train = X_train.copy(); X_val = X_val.copy(); X_test = X_test.copy()
for frame in (X_train, X_val, X_test):
    frame['Game_Clean'] = frame['Game'].where(frame['Game'].isin(top_games), 'Other')
    top_styles = ['Multiplayer - online - with real life friends',
                  'Multiplayer - online - with strangers',
                  'Multiplayer - online - with online acquaintances or teammates', 'Singleplayer']
    frame['Playstyle_Clean'] = frame['Playstyle'].where(frame['Playstyle'].isin(top_styles), 'Other')
    top_why = ['having fun', 'improving', 'winning', 'relaxing']
    frame['whyplay_Clean'] = frame['whyplay'].where(frame['whyplay'].isin(top_why), 'Other')
    frame['SPIN_SWL_Ratio'] = frame['SPIN_T'] / (frame['SWL_T'] + 1.0)
    frame.drop(columns=['Game', 'Playstyle', 'whyplay'], inplace=True)

numeric_cols = ['Hours', 'streams', 'Age', 'SWL_T', 'SPIN_T', 'Narcissism', 'SPIN_SWL_Ratio']
categorical_cols = ['Platform', 'Gender', 'Work', 'Degree', 'Game_Clean', 'Playstyle_Clean', 'whyplay_Clean']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                      ('scaler', RobustScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols)
])

## 6.1.5 Pelatihan baseline dan hyperparameter awal

| Hyperparameter | Nilai | Alasan |
|---|---:|---|
| `solver` | `lbfgs` | Optimizer stabil untuk Logistic Regression multifeature |
| `max_iter` | 2.000 | Memastikan konvergensi |
| `class_weight` | `balanced` | Menambah penalti kesalahan pada kelas risiko tinggi |
| regularisasi | L2 bawaan, `C=1.0` | Mengendalikan kompleksitas sambil mempertahankan baseline standar |
| threshold | 0,50 | Titik keputusan awal yang konsisten untuk perbandingan |

In [4]:
baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        solver='lbfgs', max_iter=2000, class_weight='balanced',
        C=1.0, random_state=RANDOM_SEED
    ))
])
baseline.fit(X_train, y_train)
print(f'Baseline berhasil dilatih pada {len(X_train):,} sampel train.')

Baseline berhasil dilatih pada 7,788 sampel train.


## 6.1.6 Hasil pada validation set

Metrik utama untuk masalah tidak seimbang adalah Recall kelas positif, Macro-F1, ROC-AUC, dan PR-AUC. Akurasi dilaporkan sebagai konteks saja.

In [5]:
val_pred = baseline.predict(X_val)
val_proba = baseline.predict_proba(X_val)[:, 1]

metrics_baseline = pd.DataFrame({
    'metric': ['Accuracy', 'Precision (positive)', 'Recall (positive)', 'F1 (positive)', 'Macro-F1', 'ROC-AUC', 'PR-AUC'],
    'score': [
        accuracy_score(y_val, val_pred),
        precision_score(y_val, val_pred, zero_division=0),
        recall_score(y_val, val_pred),
        f1_score(y_val, val_pred, zero_division=0),
        f1_score(y_val, val_pred, average='macro'),
        roc_auc_score(y_val, val_proba),
        average_precision_score(y_val, val_proba),
    ]
})
print(metrics_baseline.to_string(index=False))
print('\nConfusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_val, val_pred))

              metric     score
            Accuracy  0.721109
Precision (positive)  0.347486
   Recall (positive)  0.689579
    F1 (positive)     0.462110
            Macro-F1  0.636931
             ROC-AUC  0.782539
              PR-AUC  0.488877

Confusion matrix [[TN, FP], [FN, TP]]:
[[1561  584]
 [ 140  311]]


### Interpretasi baseline

Baseline menemukan 311 dari 451 responden berisiko tinggi pada validation set (Recall **0,690**), tetapi menghasilkan 584 positive false alarms. Trade-off ini konsisten dengan penggunaan bobot kelas: dalam konteks skrining dini, kehilangan responden berisiko (false negative) lebih mahal daripada menandai seseorang untuk pemeriksaan lanjut. ROC-AUC **0,783** menunjukkan kemampuan pemeringkatan yang baik di atas tebakan acak.

Belum ada klaim performa final: hold-out test tetap tersegel sampai desain model dan threshold diputuskan.